## Concise, barebones workflow for getting to a TSM Line

#### Imports

In [ ]:
import sys, os
basedir = ''
if "__file__" in globals(): basedir = os.path.dirname(__file__)
sys.path.insert(0, os.path.join(basedir, os.path.pardir, os.path.pardir, 'python'))

In [ ]:
import pandas as pd
import numpy as np
import scipy as sci
import matplotlib.pyplot as pl
import matplotlib.image as img
import subprocess
import pathlib
import pyvista as pv
import copy

import fenics_sz.utils
output_folder = pathlib.Path(os.path.join(basedir, "output"))
output_folder.mkdir(exist_ok=True, parents=True)

In [ ]:
from fenics_sz.sz_problems.sz_slab import create_slab, plot_slab
from fenics_sz.sz_problems.sz_geometry import create_sz_geometry
from fenics_sz.sz_problems.sz_steady_dislcreep import SteadyDislSubductionProblem
from fenics_sz.sz_problems.sz_tdep_dislcreep import TDDislSubductionProblem
from fenics_sz.sz_problems.sz_params import default_params, allsz_params

In [ ]:
from fenics_sz.fluid_release.perple_x_integration import get_PT_data_from_tabs, plot_PT_data
import fenics_sz.fluid_release.get_PT_curves 

#### Read Perple_X data

In [ ]:
DMM_data = get_PT_data_from_tabs('DMMdamp_25')
uvolcs_data = get_PT_data_from_tabs('upvolc_25')
lvolcs_data = get_PT_data_from_tabs('lovolc_25')
dikes_data = get_PT_data_from_tabs('dike_25')
gabbros_data = get_PT_data_from_tabs('gabbro_25')

#### Create and Solve SZ

In [ ]:
resscale = 5.0
alaskan_peninsula_dict = allsz_params["01_Alaska_Peninsula"]
alaskan_peninsula_slab = create_slab(alaskan_peninsula_dict['xs'], alaskan_peninsula_dict['ys'], resscale, alaskan_peninsula_dict['lc_depth'])
plot_slab(alaskan_peninsula_slab)

In [ ]:
alaskan_peninsula_dict['xs'][-1]

In [ ]:
alaskan_peninsula_geom = create_sz_geometry(alaskan_peninsula_slab, resscale, alaskan_peninsula_dict['sztype'], alaskan_peninsula_dict['io_depth'], alaskan_peninsula_dict['extra_width'], 
                             alaskan_peninsula_dict['coast_distance'], alaskan_peninsula_dict['lc_depth'], alaskan_peninsula_dict['uc_depth'])
alaskan_peninsula_sz = TDDislSubductionProblem(alaskan_peninsula_geom, **alaskan_peninsula_dict)

alaskan_peninsula_sz.solve(alaskan_peninsula_dict['As'], dt=0.05, theta=0.5, rtol=1.e-1, verbosity=1)

plotter = pv.Plotter()
fenics_sz.utils.plot.plot_scalar(alaskan_peninsula_sz.T_i, plotter=plotter, scale=alaskan_peninsula_sz.T0, gather=True, cmap='coolwarm', scalar_bar_args={'title': 'Temperature (deg C)', 'bold':True})
# fenics_sz.utils.plot.plot_vector_glyphs(alaskan_peninsula_sz.vw_i, plotter=plotter, gather=True, factor=0.1, color='k', scale=fenics_sz.utils.mps_to_mmpyr(alaskan_peninsula_sz.v0))
# fenics_sz.utils.plot.plot_vector_glyphs(alaskan_peninsula_sz.vs_i, plotter=plotter, gather=True, factor=0.1, color='k', scale=fenics_sz.utils.mps_to_mmpyr(alaskan_peninsula_sz.v0))
alaskan_peninsula_geom.pyvistaplot(plotter=plotter, color='green', width=2)
cdpt = alaskan_peninsula_slab.findpoint('Slab::FullCouplingDepth')
fenics_sz.utils.plot.plot_points([[cdpt.x, cdpt.y, 0.0]], plotter=plotter, render_points_as_spheres=True, point_size=10.0, color='green')
fenics_sz.utils.plot.plot_show(plotter)
# fenics_sz.utils.plot.plot_save(plotter, output_folder / "{}_td_solution_resscale_{:.2f}.png".format("kamchatka", resscale))

## Workflow for producing a TSM plot

* generate P-T-H2O% plots for the minerologies in the slab

* get points along slab in a slab-tangent slab-normal coordinate system
* probe the depths and temperatures at those points
* convert the depths to pressures at each point
* use the pressure-temperature data at each point to get a hydration % for each point
* (use groups of four points to get an averaged H2O % for the cell the points create?)
* calculate the area of each cell
* convert cell area to weight per meter of trench
* convert weight per meter of trench to water weight
* wherever there is a change in water content along a slab normal path, catalog the average depth of the two cells across which the change occurs.


### Test to make sure a slab orthogonal grid can be made

In [ ]:
def in_domain(sz, point):
    return ((0 < point[0] < int(sz.geom.slab_spline(1)[0])) and (int(sz.geom.slab_spline(1)[1]) < point[1] < 0))

In [ ]:
# added offset to account for sediment thickness

def get_regular_grid_attempt_5(sz, h_serp, u_res, sediment_offset):
    xys = []
    spline_dist = []
    spline_depth = []
    us = np.linspace(0,1,u_res)

    test_spline = copy.deepcopy(sz.geom.slab_spline)
    test_spline.translatenormal(-7 -h_serp -sediment_offset)
    test_xys = np.asarray([test_spline(u) + [0.0] for u in us])
    good_us = [] # list of u values to use when creating the set of xy points in slab space 
    for u in range(u_res):
        if in_domain(sz, test_xys[u]) ==True:
            good_us.append(us[u])
    print(good_us)

#upper and lower volcanics
    depth = 0 -sediment_offset
    volcanics_top = copy.deepcopy(sz.geom.slab_spline)
    xys.append(np.asarray([volcanics_top(u) + [0.0] for u in good_us]))
    spline_dist.append(np.asarray([u*volcanics_top.length for u in good_us]))
    spline_depth.append(depth)

    depth = -0.3 -sediment_offset
    volcanics_middle = copy.deepcopy(sz.geom.slab_spline)
    volcanics_middle.translatenormal(depth)
    xys.append(np.asarray([volcanics_middle(u) + [0.0] for u in good_us]))
    spline_dist.append(np.asarray([u*volcanics_middle.length for u in good_us]))
    spline_depth.append(depth)

    depth = -0.6 -sediment_offset
    volcanics_bottom = copy.deepcopy(sz.geom.slab_spline)
    volcanics_bottom.translatenormal(depth)
    xys.append(np.asarray([volcanics_bottom(u) + [0.0] for u in good_us]))
    spline_dist.append(np.asarray([u*volcanics_bottom.length for u in good_us]))
    spline_depth.append(depth)


# dikes
    depth = -1.3 -sediment_offset
    dikes_middle = copy.deepcopy(sz.geom.slab_spline)
    dikes_middle.translatenormal(depth)
    xys.append(np.asarray([dikes_middle(u) + [0.0] for u in good_us]))
    spline_dist.append(np.asarray([u*dikes_middle.length for u in good_us]))
    spline_depth.append(depth)

    depth = -2.0 -sediment_offset
    dikes_bottom = copy.deepcopy(sz.geom.slab_spline)
    dikes_bottom.translatenormal(depth)
    xys.append(np.asarray([dikes_bottom(u) + [0.0] for u in good_us]))
    spline_dist.append(np.asarray([u*dikes_bottom.length for u in good_us]))
    spline_depth.append(depth)

# gabbros:
    for i in range(5): #1km spaced layers in gabbros (5km thick)
        depth = (-2.0 - (i+1) -sediment_offset) # start of mantle depth plus depth of increments w/n mantle
        new_spline = copy.deepcopy(sz.geom.slab_spline)
        new_spline.translatenormal(depth)
        xys.append(np.asarray([new_spline(u) + [0.0] for u in good_us]))
        spline_dist.append(np.asarray([u*new_spline.length for u in good_us]))
        spline_depth.append(depth)


# mantle:
    for i in range(h_serp): # 1km spaced lines in the serp. mantle
        depth = (-7.0 - (i+1) -sediment_offset) # start of mantle depth plus depth of increments w/n mantle
        new_spline = copy.deepcopy(sz.geom.slab_spline)
        new_spline.translatenormal(depth)
        xys.append(np.asarray([new_spline(u) + [0.0] for u in good_us]))
        spline_dist.append(np.asarray([u*new_spline.length for u in good_us]))
        spline_depth.append(depth)

    xys_as_array = np.asarray(xys)
    return xys_as_array, spline_dist, spline_depth

In [ ]:
# true 'slab-normal' coord system

def get_regular_grid_attempt_6(sz, h_serp, u_res, sediment_offset):
    
    new_xys = []
    spline_dist = []
    spline_depth = []
    us = np.linspace(0,1,u_res)

# Checking which points will fall in the domain:
    test_spline = copy.deepcopy(sz.geom.slab_spline)
    test_spline.translatenormal(-7 -h_serp -sediment_offset)
    test_xys = np.asarray([test_spline(u) + [0.0] for u in us])

# ----------------- cian's code below -----------------

    slab_xys = np.asarray([sz.geom.slab_spline(u) + [0.0] for u in us]) # xy points on the surface of the slab

    normals = np.stack([-sz.geom.slab_spline.cs(test_xys[:,0], nu=1), np.ones(u_res), np.zeros(u_res)], axis=1)
    normags = np.sqrt(np.sum(normals**2, axis=1))
    normals = (normals.T/normags).T
    # getting the normals

    new_test_xys_new = slab_xys + (-7 -h_serp -sediment_offset)*normals
    # new points are generated using vector addition
    
# ------------------------------------------------------

    good_us = [] # list of u values to use when creating the set of xy points in slab space 
    for u in range(u_res):
        if in_domain(sz, new_test_xys_new[u]) ==True:
            good_us.append(us[u])
    # print(good_us)

    good_slab_xys = np.asarray([sz.geom.slab_spline(u) + [0.0] for u in good_us]) # xy points on the surface of the slab



#upper and lower volcanics
    depth = 0 -sediment_offset
    volcanics_top = copy.deepcopy(sz.geom.slab_spline)
    xys = (np.asarray([volcanics_top(u) + [0.0] for u in good_us]))
    spline_dist.append(np.asarray([u*volcanics_top.length for u in good_us]))
    spline_depth.append(depth)

    normals = np.stack([-sz.geom.slab_spline.cs(xys[:,0], nu=1), np.ones(len(good_us)), np.zeros(len(good_us))], axis=1)
    normags = np.sqrt(np.sum(normals**2, axis=1))
    normals = (normals.T/normags).T
    new_xys.append(np.asarray(good_slab_xys + (depth)*normals))

    depth = -0.3 -sediment_offset
    volcanics_middle = copy.deepcopy(sz.geom.slab_spline)
    volcanics_middle.translatenormal(depth)
    xys = (np.asarray([volcanics_middle(u) + [0.0] for u in good_us]))
    spline_dist.append(np.asarray([u*volcanics_middle.length for u in good_us]))
    spline_depth.append(depth)

    normals = np.stack([-sz.geom.slab_spline.cs(xys[:,0], nu=1), np.ones(len(good_us)), np.zeros(len(good_us))], axis=1)
    normags = np.sqrt(np.sum(normals**2, axis=1))
    normals = (normals.T/normags).T
    new_xys.append(np.asarray(good_slab_xys + (depth)*normals))


    depth = -0.6 -sediment_offset
    volcanics_bottom = copy.deepcopy(sz.geom.slab_spline)
    volcanics_bottom.translatenormal(depth)
    xys = (np.asarray([volcanics_bottom(u) + [0.0] for u in good_us]))
    spline_dist.append(np.asarray([u*volcanics_bottom.length for u in good_us]))
    spline_depth.append(depth)

    normals = np.stack([-sz.geom.slab_spline.cs(xys[:,0], nu=1), np.ones(len(good_us)), np.zeros(len(good_us))], axis=1)
    normags = np.sqrt(np.sum(normals**2, axis=1))
    normals = (normals.T/normags).T
    new_xys.append(np.asarray(good_slab_xys + (depth)*normals))



# dikes
    depth = -1.3 -sediment_offset
    dikes_middle = copy.deepcopy(sz.geom.slab_spline)
    dikes_middle.translatenormal(depth)
    xys = (np.asarray([dikes_middle(u) + [0.0] for u in good_us]))
    spline_dist.append(np.asarray([u*dikes_middle.length for u in good_us]))
    spline_depth.append(depth)


    normals = np.stack([-sz.geom.slab_spline.cs(xys[:,0], nu=1), np.ones(len(good_us)), np.zeros(len(good_us))], axis=1)
    normags = np.sqrt(np.sum(normals**2, axis=1))
    normals = (normals.T/normags).T
    new_xys.append(np.asarray(good_slab_xys + (depth)*normals))


    depth = -2.0 -sediment_offset
    dikes_bottom = copy.deepcopy(sz.geom.slab_spline)
    dikes_bottom.translatenormal(depth)
    xys = (np.asarray([dikes_bottom(u) + [0.0] for u in good_us]))
    spline_dist.append(np.asarray([u*dikes_bottom.length for u in good_us]))
    spline_depth.append(depth)

    normals = np.stack([-sz.geom.slab_spline.cs(xys[:,0], nu=1), np.ones(len(good_us)), np.zeros(len(good_us))], axis=1)
    normags = np.sqrt(np.sum(normals**2, axis=1))
    normals = (normals.T/normags).T
    new_xys.append(np.asarray(good_slab_xys + (depth)*normals))


# gabbros:
    for i in range(5): #1km spaced layers in gabbros (5km thick)
        depth = (-2.0 - (i+1) -sediment_offset) # start of mantle depth plus depth of increments w/n mantle
        new_spline = copy.deepcopy(sz.geom.slab_spline)
        new_spline.translatenormal(depth)
        xys = (np.asarray([new_spline(u) + [0.0] for u in good_us]))
        spline_dist.append(np.asarray([u*new_spline.length for u in good_us]))
        spline_depth.append(depth)

        normals = np.stack([-sz.geom.slab_spline.cs(xys[:,0], nu=1), np.ones(len(good_us)), np.zeros(len(good_us))], axis=1)
        normags = np.sqrt(np.sum(normals**2, axis=1))
        normals = (normals.T/normags).T
        new_xys.append(np.asarray(good_slab_xys + (depth)*normals))



# mantle:
    for i in range(h_serp): # 1km spaced lines in the serp. mantle
        depth = (-7.0 - (i+1) -sediment_offset) # start of mantle depth plus depth of increments w/n mantle
        new_spline = copy.deepcopy(sz.geom.slab_spline)
        new_spline.translatenormal(depth)
        xys = (np.asarray([new_spline(u) + [0.0] for u in good_us]))
        spline_dist.append(np.asarray([u*new_spline.length for u in good_us]))
        spline_depth.append(depth)

        normals = np.stack([-sz.geom.slab_spline.cs(xys[:,0], nu=1), np.ones(len(good_us)), np.zeros(len(good_us))], axis=1)
        normags = np.sqrt(np.sum(normals**2, axis=1))
        normals = (normals.T/normags).T
        new_xys.append(np.asarray(good_slab_xys + (depth)*normals))


    xys_as_array = np.asarray(new_xys)
    return xys_as_array, spline_dist, spline_depth

In [ ]:
dist_res = 100
h_serp = 2
sediment_offset = alaskan_peninsula_dict["z15"]

In [ ]:
regular_points, spline_dist, layer_depths = get_regular_grid_attempt_6(alaskan_peninsula_sz, h_serp, dist_res, sediment_offset)

In [ ]:
print(len(regular_points))
print(len(regular_points[0])) 
print(len(regular_points[0][0]))

print(type(regular_points))
print(type(regular_points[0]))
print((regular_points[11][43]))


In [ ]:
# print(len(layer_depths))

for i in range(len(layer_depths)):
    print(layer_depths[i])

In [ ]:
print(len(regular_points))
print(len(regular_points[0])) #thinner h_serp gives you more valid slab dists. to wok with
print(len(regular_points[0][0]))

print(type(regular_points))
print(type(regular_points[0]))
print((regular_points[11][43]))


In [ ]:
print(len(spline_dist))
print(len(spline_dist[0]))

for i in range(len(spline_dist)):
    print(spline_dist[i][94])

#### Check to make sure points fall in a grid

In [ ]:
plotter = pv.Plotter()
fenics_sz.utils.plot.plot_scalar(alaskan_peninsula_sz.T_i, plotter=plotter, scale=alaskan_peninsula_sz.T0, gather=True, cmap='coolwarm', scalar_bar_args={'title': 'Temperature (deg C)', 'bold':True})
alaskan_peninsula_geom.pyvistaplot(plotter=plotter, color='green', width=2)
cdpt = alaskan_peninsula_slab.findpoint('Slab::FullCouplingDepth')
fenics_sz.utils.plot.plot_points([[cdpt.x, cdpt.y, 0.0]], plotter=plotter, render_points_as_spheres=True, point_size=10.0, color='green')

for i in range(len(regular_points)):
    for j in range(len(regular_points[i])):            
        fenics_sz.utils.plot.plot_points([[regular_points[i][j][0], regular_points[i][j][1], 0.0]], plotter=plotter, point_size=0.5, color='black')

fenics_sz.utils.plot.plot_show(plotter)

#### Precalculating temperatures

In [ ]:
print(len(regular_points))
print(len(regular_points[0]))
print(len(regular_points[0][0]))

print(type(regular_points))
print(type(regular_points[0]))
print(type(regular_points[0][0]))


In [ ]:
def get_interpolator(perple_x_data):
    PT_points = (perple_x_data[0], perple_x_data[1])
    predict = sci.interpolate.RegularGridInterpolator(PT_points, perple_x_data[2], method = 'linear')
    return predict

    #create an interpolator object given the p, t, h2o perplex output

def predict_h2o(interpolator, t, p):
    # print("Temp: ", t)
    # print("Pressure: ", p)
    if t < 200:
        return interpolator([p, 200])
    # case for if the temperatures fall outside the bounds of the data used to generate the interpolator

    else:
        return (interpolator([p, t]))

    #predict the %wt hydration at a point given the pressure and temperature

In [ ]:
uvolcs_interp = get_interpolator(uvolcs_data)
lvolcs_interp = get_interpolator(lvolcs_data)
dikes_interp = get_interpolator(dikes_data)
gabbros_interp = get_interpolator(gabbros_data)
damp_DMM_interp = get_interpolator(DMM_data)

In [ ]:
ts = []
zs = []
ps = []

for i in range(len(regular_points)):
    cinds, cells = fenics_sz.utils.mesh.get_cell_collisions(regular_points[i], alaskan_peninsula_sz.mesh)
    t, z, p = (alaskan_peninsula_sz.T_i.eval(regular_points[i], cells)[:,0] + (0.3 * -regular_points[i][:,1]), -regular_points[i][:,1], (-regular_points[i][:,1])*1000 * 3300 * 9.81 *1e-9)
    ts.append(t)
    zs.append(z)
    ps.append(p)

# ts, zs, ps, are all 2d arrays. first index gives the layer, second index gives the u.
# value is just value of the point. 

In [ ]:
print(len(zs))
print(len(zs[0]))

for i in range(len(zs)):
    print(zs[i][94])

## The Cell class

In [ ]:
# pass in a PT H2O interpolator when initializing the class, instead of the perpleX data
# one interpolator per lithology

# i can also try structuring it such that the temp of each vertex is passed in as a parameter.
# this way, mesh-point collisions and temperature evals are only done once per point, as opposed to four times


class Cell:
    def __init__(self, sz, slab_depth_1, slab_depth_2, slab_dist_1, slab_dist_2, slab_dist_3, slab_dist_4,
                 z1, z2, z3, z4, p1, p2, p3, p4, t1, t2, t3, t4, interpolator):
        self.sz = sz
        self.slab_depth_1 = slab_depth_1
        self.slab_depth_2 = slab_depth_2
        self.slab_dist_1 = slab_dist_1
        self.slab_dist_2 = slab_dist_2
        self.slab_dist_3 = slab_dist_3
        self.slab_dist_4 = slab_dist_4

        self.pressures = np.array([p1, p2, p3, p4])
        self.temps = np.array([t1, t2, t3, t4])
        self.vertex_depths = np.array([z1, z2, z3, z4])
        self.interpolator = interpolator
        self._water_pct = None
        self._hydrations = []
    
    def get_area(self):
        return((self.slab_depth_2 - self.slab_depth_1) * (((self.slab_dist_2 - self.slab_dist_1) + (self.slab_dist_4 - self.slab_dist_3)) / 2))
    # very height times avg. length approx

    def get_depth(self):
        return(sum(self.vertex_depths)/4)

    def get_high_depth(self):
        pass

    def get_low_depth(self):
        pass

    def get_temp(self):
        return(sum(self.temps)/4)

    def get_water_pct(self):
        if self._water_pct is None:

            for i in range(len(self.temps)):
                if self.temps[i] < 200:
                    self._hydrations.append(self.interpolator([self.pressures[i], 200]))
                # case for if the temperatures fall outside the bounds of the data used to generate the interpolator

                else:
                    self._hydrations.append(self.interpolator([self.pressures[i], self.temps[i]]))

            self._water_pct = sum(self._hydrations)/4
        return self._water_pct

    def get_water_wt(self):
        return (self.get_water_pct() * 1e-2 * self.get_area() * 3300 * 1e-3)
        # units documentation:
        # first putting percents in decimal form
        # multiplying by area to get km^2

        # multiply density by 1e9 to get the density in units of kg/km^3
        # multiply current value by 1e-9 to convert from kg to Tg
        # two conversions above cancel out

        # left with units of Tg/km
        # multiply by 1e-3 to get Tg/m



    def __repr__(self):
        pass


### Initializing cells by indexing over regular points

In [ ]:
print(len(regular_points))
print(len(regular_points[0]))
print(len(regular_points[0][0]))

print(type(regular_points))
print(type(regular_points[0]))
print(type(regular_points[0][0]))

In [ ]:
alaskan_peninsula_cells = []
interp = None
for i in range(len(layer_depths)-1):
    layer_cells = []

    if layer_depths[i+1] >= -0.3 -sediment_offset:
        interp = uvolcs_interp
    elif layer_depths[i+1] >= -0.6 -sediment_offset:
        interp = lvolcs_interp
    elif layer_depths[i+1] >= -2 -sediment_offset:
        interp = dikes_interp
    elif layer_depths[i+1] >= -7 -sediment_offset:
        interp = gabbros_interp
    elif layer_depths[i+1] >= -7 - h_serp -sediment_offset:
        interp = damp_DMM_interp
    else:
        raise Exception("Improper cell depth")



    for j in range(len(regular_points[i])-1): # problem with area approx is that spline dist varies between layers (taking average of spline dist btwn cell's 2 defining layers might make it a bit better)
        cell = Cell(alaskan_peninsula_sz, layer_depths[i], layer_depths[i+1], spline_dist[i][j], spline_dist[i][j+1], spline_dist[i+1][j], spline_dist[i+1][j+1],
                    zs[i][j], zs[i][j+1], zs[i+1][j], zs[i+1][j+1], ps[i][j], ps[i][j+1], ps[i+1][j], ps[i+1][j+1], 
                    ts[i][j], ts[i][j+1], ts[i+1][j], ts[i+1][j+1], interp)
        # might want to consider going back to passing in an interpolator object, and doing the hydration prediction within the class
        layer_cells.append(cell)
    alaskan_peninsula_cells.append(layer_cells)

#### some code to check what interpolators are actually being passed into the cells:

In [ ]:

interp = None
for i in range(len(regular_points)-1):
    if layer_depths[i+1] >= -0.3 -sediment_offset:
        interp = "uvolcs_interp"
    elif layer_depths[i+1] >= -0.6 -sediment_offset:
        interp = "lvolcs_interp"
    elif layer_depths[i+1] >= -2 -sediment_offset:
        interp = "dikes_interp"
    elif layer_depths[i+1] >= -7 -sediment_offset:
        interp = "gabbros_interp"
    elif layer_depths[i+1] >= -7 - h_serp -sediment_offset:
        interp = "damp_DMM_interp"
    else:
        raise Exception("Improper cell depth")
       
    print("Depth of Cell Bottom: " , layer_depths[i+1], " -------- ",  "Interpolator: " , interp)


#### Getting cell hydrations:

In [ ]:
cell_hydrations = []
for i in range(len(alaskan_peninsula_cells)):
    layer_cell_hydrations = []
    for j in range(len(alaskan_peninsula_cells[i])):
        layer_cell_hydrations.append(alaskan_peninsula_cells[i][j].get_water_pct()[0])
    cell_hydrations.append(layer_cell_hydrations)

In [ ]:
print(len(cell_hydrations))
print(len(cell_hydrations[0]))
print((cell_hydrations[0][0]))

In [ ]:
print(len(alaskan_peninsula_cells))
print(len(alaskan_peninsula_cells[0]))
# print(len(alaskan_peninsula_cells[0][0]))

In [ ]:
fig, ax = pl.subplots()

slab_surf_dist = [] # distance along slab *surface* , not along the layer
for i in range(len(spline_dist[0])):
    slab_surf_dist.append(spline_dist[0][i])


c = ax.pcolor(slab_surf_dist, layer_depths, cell_hydrations, cmap='Blues')
ax.set_title('alaskan_peninsula Cell HYDRATIONS; 5x Slab-Normal Exageration')
ax.set_xlabel('Distance along slab surface (km)')
ax.set_ylabel('Distance normal to slab surface (km)')

slab_normal_exag = 5
ax.set_box_aspect((-layer_depths[-1] / slab_surf_dist[-1]) * slab_normal_exag)


fig.colorbar(c, ax=ax)
pl.show()

#### Cell Temperatures

In [ ]:
# this prints temperatures along a slab-normal line!
for i in range(len(alaskan_peninsula_cells)):
    print(alaskan_peninsula_cells[i][80].get_temp())

cell_temps = []
for i in range(len(alaskan_peninsula_cells)):
    layer_cell_temps = []
    for j in range(len(alaskan_peninsula_cells[i])):
        layer_cell_temps.append(alaskan_peninsula_cells[i][j].get_temp())
    cell_temps.append(layer_cell_temps)

In [ ]:
fig, ax = pl.subplots()

slab_surf_dist = [] # distance along slab *surface* , not along the layer
for i in range(len(spline_dist[0])):
    slab_surf_dist.append(spline_dist[0][i])


c = ax.pcolor(slab_surf_dist, layer_depths, cell_temps, cmap='RdBu_r')
ax.set_title('alaskan_peninsula Cell Temps; 5x Slab-Normal Exageration')
ax.set_xlabel('Distance along slab surface (km)')
ax.set_ylabel('Distance normal to slab surface (km)')

slab_normal_exag = 5
ax.set_box_aspect((-layer_depths[-1] / slab_surf_dist[-1]) * slab_normal_exag)


fig.colorbar(c, ax=ax)
pl.show()


#### Disallowing rehydration

In [ ]:
def remove_rehydration(cell_hydrations):
    for i in range(len(cell_hydrations)):
        for j in range(len(cell_hydrations[i])-1):
            if cell_hydrations[i][j+1] > cell_hydrations[i][j]:
                cell_hydrations[i][j+1] = cell_hydrations[i][j]
    # FIXME more of a warning: this function transforms the input, it doesn't create a new variable
    # that is, when used, an initial hydrations array that allows for rehydration is not retained
    return cell_hydrations

In [ ]:
alaskan_peninsula_cell_hydrations_no_rehydration = remove_rehydration(cell_hydrations)

In [ ]:
fig, ax = pl.subplots()

slab_surf_dist = [] # distance along slab *surface* , not along the layer
for i in range(len(spline_dist[0])):
    slab_surf_dist.append(spline_dist[0][i])


c = ax.pcolor(slab_surf_dist, layer_depths, cell_hydrations, cmap='Blues') 
# this should actually be plotting hydrations considering allowing for rehydration...
ax.set_title('alaskan_peninsula Cell Hydration NO REHYDRATION; 5x Slab-Normal Exageration')
ax.set_xlabel('Distance along slab surface (km)')
ax.set_ylabel('Distance normal to slab surface (km)')

slab_normal_exag = 5
ax.set_box_aspect((-layer_depths[-1] / slab_surf_dist[-1]) * slab_normal_exag)


fig.colorbar(c, ax=ax)
pl.show()

#### Water Loss

In [ ]:
def get_water_loss(cells, cell_hydrations):
    slab_losses_and_depths = []
    for i in range(len(cell_hydrations)):
        for j in range(len(cell_hydrations[i])-1):
            if (cell_hydrations[i][j+1] < cell_hydrations[i][j]):
                # should I check against an epsilon instead?

                # losses_and_depths = [cell_hydrations[i][j] - cell_hydrations[i][j+1] , ((cells[i][j].get_depth() + cells[i][j+1].get_depth()) / 2)]


                #This line stores total water lost, not water pct. water pct needs to be used for comparison, not water loss (I think...)
                losses_and_depths = [cells[i][j+1].get_water_wt() - cells[i][j].get_water_wt() , ((cells[i][j].get_depth() + cells[i][j+1].get_depth()) / 2)]
                
                #FIXME depths are currently global depths, not depths to surface of the slab

                # water_losses.append(cell_hydrations[i][j] - cell_hydrations[i][j+1])
                # water_loss_depths.append(((cells[i][j].get_depth() + cells[i][j+1].get_depth()) / 2))
                slab_losses_and_depths.append(losses_and_depths)

    return slab_losses_and_depths


In [ ]:
alaskan_peninsula_water_losses_and_depths = get_water_loss(alaskan_peninsula_cells, alaskan_peninsula_cell_hydrations_no_rehydration)

In [ ]:
print(len(alaskan_peninsula_water_losses_and_depths))
print(len(alaskan_peninsula_water_losses_and_depths[0]))

# for i in range(len(alaskan_peninsula_water_losses_and_depths)):
#     print(alaskan_peninsula_water_losses_and_depths[i][1])

sorted_alaskan_peninsula_water_losses_and_depths = sorted(alaskan_peninsula_water_losses_and_depths, key=lambda l:l[1])
print("list sorted")


# for i in range(len(sorted_alaskan_peninsula_water_losses_and_depths)):
#     print(sorted_alaskan_peninsula_water_losses_and_depths[i][1]) #check that depths are actually sorted


for i in range(len(sorted_alaskan_peninsula_water_losses_and_depths)):
    print(sorted_alaskan_peninsula_water_losses_and_depths[i][0]) #check what the water losses are (should be in Tg/m)


In [ ]:
fig, ax = pl.subplots()
ax.plot([row[0] for row in sorted_alaskan_peninsula_water_losses_and_depths], [row[1] for row in sorted_alaskan_peninsula_water_losses_and_depths])
ax.yaxis.set_inverted(True)  # inverted axis with autoscaling

ax.set_title('alaskan_peninsula water loss (delta (Tg/m)) as a function of depth (km)')
ax.set_xlabel('Tg/m lost')
ax.set_ylabel('Depth where water loss occurs (km)')

pl.show()

### Converting to Tokyo subway map units: Tg/MYr/m

#### Cumulative sum

In [ ]:
# performing cumulative sum on Tg/m loss array to check if shape aligns w literature

cum_sum_array = []
cum_sum = 0
for i in range(len(sorted_alaskan_peninsula_water_losses_and_depths)):
    cum_sum += sorted_alaskan_peninsula_water_losses_and_depths[i][0]
    print(cum_sum)
    cum_sum_array.append(cum_sum[0])
    print(cum_sum_array)

In [ ]:
fig, ax = pl.subplots()
ax.plot(cum_sum_array, [row[1] for row in sorted_alaskan_peninsula_water_losses_and_depths])
ax.yaxis.set_inverted(True)  # inverted axis with autoscaling

ax.set_title('alaskan_peninsula CUMULATIVE water loss (delta (Tg/m)) as a function of depth (km)')
ax.set_xlabel('Tg/m lost')
ax.set_ylabel('Depth where water loss occurs (km)')



pl.show()

#### Conversion to per unit time

In [ ]:
print(alaskan_peninsula_dict['Vs'])

In [ ]:
alaskan_peninsula_distance_increment = alaskan_peninsula_sz.geom.slab_spline.length / dist_res
# (in km) slab-tangent distance between two st points on the surface of the alaskan_peninsula slab

print(alaskan_peninsula_distance_increment)

In [ ]:
time_standarized_losses = []

for i in range(len(sorted_alaskan_peninsula_water_losses_and_depths)):
    time_standarized_losses.append(sorted_alaskan_peninsula_water_losses_and_depths[i][0][0] * alaskan_peninsula_dict['Vs'] / alaskan_peninsula_distance_increment)

In [ ]:
print(len(time_standarized_losses))

In [ ]:
fig, ax = pl.subplots()
ax.plot(time_standarized_losses, [row[1] for row in sorted_alaskan_peninsula_water_losses_and_depths])
ax.yaxis.set_inverted(True)  # inverted axis with autoscaling

ax.set_title('alaskan_peninsula time-standardized water loss (delta (Tg/MYr/m)) as a function of depth (km)')
ax.set_xlabel('Tg/MYr/m lost')
ax.set_ylabel('Depth where water loss occurs (km)')



pl.show()

In [ ]:
cum_sum_time_standardized_array = []
cum_sum_time_standardized = 0
for i in range(len(time_standarized_losses)):
    cum_sum_time_standardized += time_standarized_losses[i]
    print(cum_sum_time_standardized)
    cum_sum_time_standardized_array.append(cum_sum_time_standardized)
    print(cum_sum_time_standardized_array)


In [ ]:
fig, ax = pl.subplots()
ax.plot(cum_sum_time_standardized_array, [row[1] for row in sorted_alaskan_peninsula_water_losses_and_depths])
ax.yaxis.set_inverted(True)  # inverted axis with autoscaling

ax.set_title('True Slab Normal Coordinate: alaskan_peninsula TSM line')
ax.set_xlabel('Tg/MYr/m lost')
ax.set_ylabel('Depth where water loss occurs (km)')

ax.set_box_aspect(2.5)


pl.show()

print("Total water loss: " , cum_sum_time_standardized)
fig.savefig(output_folder / "true_slab_normal_alaskan_peninsula_TSM")

Still need to correct depths to be depth to surface of slab instead of depth where the water loss occurs

slab.intersectx(x)[1]

#### Sidebar: splitting up water loss tracking by lithology

In [ ]:
#FIXME this function is very specifc
# to how I've manually defined the layers of cells in each lithology

def sorted_water_loss_by_layer(cells, cell_hydrations):
    uvolc_losses_and_depths = []
    lvolc_losses_and_depths = []
    dike_losses_and_depths = []
    gabbros_losses_and_depths = []
    mantle_losses_and_depths = []

    for j in range(len(cell_hydrations[0])-1):
        if (cell_hydrations[0][j+1] < cell_hydrations[0][j]):

            #This line stores total water lost, not water pct. water pct needs to be used for comparison, not water loss (I think...)
            losses_and_depths = [cells[0][j+1].get_water_wt() - cells[0][j].get_water_wt(),
                                  ((cells[0][j].get_depth() + cells[0][j+1].get_depth()) / 2)]
            
            #FIXME depths are currently global depths, not depths to surface of the slab

            uvolc_losses_and_depths.append(losses_and_depths)
            lvolc_losses_and_depths.append(losses_and_depths)
            dike_losses_and_depths.append(losses_and_depths)
            gabbros_losses_and_depths.append(losses_and_depths)
            mantle_losses_and_depths.append(losses_and_depths)


    for j in range(len(cell_hydrations[1])-1):
        if (cell_hydrations[1][j+1] < cell_hydrations[1][j]):
            losses_and_depths = [cells[1][j+1].get_water_wt() - cells[1][j].get_water_wt(),
                                  ((cells[1][j].get_depth() + cells[1][j+1].get_depth()) / 2)]
            lvolc_losses_and_depths.append(losses_and_depths)
            dike_losses_and_depths.append(losses_and_depths)
            gabbros_losses_and_depths.append(losses_and_depths)
            mantle_losses_and_depths.append(losses_and_depths)

    for j in range(len(cell_hydrations[2])-1):
        if (cell_hydrations[2][j+1] < cell_hydrations[2][j]):
            losses_and_depths = [cells[2][j+1].get_water_wt() - cells[2][j].get_water_wt(),
                                  ((cells[2][j].get_depth() + cells[2][j+1].get_depth()) / 2)]
            dike_losses_and_depths.append(losses_and_depths)
            gabbros_losses_and_depths.append(losses_and_depths)
            mantle_losses_and_depths.append(losses_and_depths)

    for j in range(len(cell_hydrations[3])-1):
        if (cell_hydrations[3][j+1] < cell_hydrations[3][j]):
            losses_and_depths = [cells[3][j+1].get_water_wt() - cells[3][j].get_water_wt(),
                                  ((cells[3][j].get_depth() + cells[3][j+1].get_depth()) / 2)]
            dike_losses_and_depths.append(losses_and_depths)
            gabbros_losses_and_depths.append(losses_and_depths)
            mantle_losses_and_depths.append(losses_and_depths)

    for i in range(5):
        for j in range(len(cell_hydrations[i+4])-1):
            if (cell_hydrations[i+4][j+1] < cell_hydrations[i+4][j]):
                losses_and_depths = [cells[i+4][j+1].get_water_wt() - cells[i+4][j].get_water_wt(),
                                      ((cells[i+4][j].get_depth() + cells[i+4][j+1].get_depth()) / 2)]

                gabbros_losses_and_depths.append(losses_and_depths)
                mantle_losses_and_depths.append(losses_and_depths)

    for i in range(2):
        for j in range(len(cell_hydrations[i+9])-1):
            if (cell_hydrations[i+9][j+1] < cell_hydrations[i+9][j]):
                losses_and_depths = [cells[i+9][j+1].get_water_wt() - cells[i+9][j].get_water_wt(),
                                      ((cells[i+9][j].get_depth() + cells[i+9][j+1].get_depth()) / 2)]

                mantle_losses_and_depths.append(losses_and_depths)



    
    return sorted(uvolc_losses_and_depths, key=lambda l:l[1]), sorted(lvolc_losses_and_depths, key=lambda l:l[1]), sorted(dike_losses_and_depths, key=lambda l:l[1]), sorted(gabbros_losses_and_depths, key=lambda l:l[1]), sorted(mantle_losses_and_depths, key=lambda l:l[1]),


In [ ]:
uvolc_losses_and_depths, lvolc_losses_and_depths, dike_losses_and_depths, gabbros_losses_and_depths, mantle_losses_and_depths = sorted_water_loss_by_layer(alaskan_peninsula_cells, alaskan_peninsula_cell_hydrations_no_rehydration)

In [ ]:
print(len(uvolc_losses_and_depths))
print(len(lvolc_losses_and_depths))
print(len(dike_losses_and_depths))
print(len(gabbros_losses_and_depths))
print(len(mantle_losses_and_depths))

print("uvolcs:")
for i in range(len(uvolc_losses_and_depths)):
    print(uvolc_losses_and_depths[i])

print("mantle")
for i in range(len(mantle_losses_and_depths)):
    print(mantle_losses_and_depths[i])

In [ ]:
alaskan_peninsula_distance_increment = alaskan_peninsula_sz.geom.slab_spline.length / dist_res
# (in km) slab-tangent distance between two st points on the surface of the alaskan_peninsula slab

print(alaskan_peninsula_distance_increment)

In [ ]:
time_standarized_uvolc_losses = []
time_standarized_lvolc_losses = []
time_standarized_dike_losses = []
time_standarized_gabbros_losses = []
time_standarized_mantle_losses = []

for i in range(len(uvolc_losses_and_depths)):
    time_standarized_uvolc_losses.append(uvolc_losses_and_depths[i][0][0] * alaskan_peninsula_dict['Vs'] / alaskan_peninsula_distance_increment)

for i in range(len(lvolc_losses_and_depths)):
    time_standarized_lvolc_losses.append(lvolc_losses_and_depths[i][0][0] * alaskan_peninsula_dict['Vs'] / alaskan_peninsula_distance_increment)

for i in range(len(dike_losses_and_depths)):
    time_standarized_dike_losses.append(dike_losses_and_depths[i][0][0] * alaskan_peninsula_dict['Vs'] / alaskan_peninsula_distance_increment)

for i in range(len(gabbros_losses_and_depths)):
    time_standarized_gabbros_losses.append(gabbros_losses_and_depths[i][0][0] * alaskan_peninsula_dict['Vs'] / alaskan_peninsula_distance_increment)

for i in range(len(mantle_losses_and_depths)):
    time_standarized_mantle_losses.append(mantle_losses_and_depths[i][0][0] * alaskan_peninsula_dict['Vs'] / alaskan_peninsula_distance_increment)


In [ ]:
print(len(time_standarized_uvolc_losses))
print(len(time_standarized_lvolc_losses))
print(len(time_standarized_dike_losses))
print(len(time_standarized_gabbros_losses))
print(len(time_standarized_mantle_losses))

In [ ]:
cum_sum_array_uvolc = []
cum_sum_array_lvolc = []
cum_sum_array_dike = []
cum_sum_array_gabbros = []
cum_sum_array_mantle = []

cum_sum = 0
for i in range(len(time_standarized_uvolc_losses)):
    cum_sum += time_standarized_uvolc_losses[i]
    # print(cum_sum)
    cum_sum_array_uvolc.append(cum_sum)
    # print(cum_sum_array_uvolc)

cum_sum = 0
for i in range(len(time_standarized_lvolc_losses)):
    cum_sum += time_standarized_lvolc_losses[i]
    cum_sum_array_lvolc.append(cum_sum)


cum_sum = 0
for i in range(len(time_standarized_dike_losses)):
    cum_sum += time_standarized_dike_losses[i]
    cum_sum_array_dike.append(cum_sum)


cum_sum = 0
for i in range(len(time_standarized_gabbros_losses)):
    cum_sum += time_standarized_gabbros_losses[i]
    cum_sum_array_gabbros.append(cum_sum)


cum_sum = 0
for i in range(len(time_standarized_mantle_losses)):
    cum_sum += time_standarized_mantle_losses[i]
    cum_sum_array_mantle.append(cum_sum)

In [ ]:
fig, ax = pl.subplots()
ax.plot(cum_sum_time_standardized_array, [row[1] for row in sorted_alaskan_peninsula_water_losses_and_depths])

ax.plot(cum_sum_array_uvolc, [row[1] for row in uvolc_losses_and_depths], label = "upper_volcs")
ax.plot(cum_sum_array_lvolc, [row[1] for row in lvolc_losses_and_depths], label = "lower_volcs")
ax.plot(cum_sum_array_dike, [row[1] for row in dike_losses_and_depths], label = "dikes")
ax.plot(cum_sum_array_gabbros, [row[1] for row in gabbros_losses_and_depths], label = "gabbros")
ax.plot(cum_sum_array_mantle, [row[1] for row in mantle_losses_and_depths], label = "mantle")


ax.yaxis.set_inverted(True)  # inverted axis with autoscaling

ax.set_title('True Slab Normal Coordinate: alaskan_peninsula TSM line')
ax.set_xlabel('Tg/MYr/m lost')
ax.set_ylabel('Depth where water loss occurs (km)')

ax.set_box_aspect(2.5)
ax.legend()


pl.show()

print("Total water loss: " , cum_sum_time_standardized)
fig.savefig(output_folder / "layer_seperated_true_slab_normal_alaskan_peninsula_TSM")

### Comparsion with Abers et al.

In [ ]:
print(basedir)

In [ ]:
fig = pl.figure(figsize=(20,10))
axi = fig.add_subplot(1,1,1)

img = pl.imread("figures/abers_alaskan_peninsula.png") # change figure
ip = axi.imshow(img)
axi.axis('off')
ax = axi.inset_axes([0.19,0.018,0.405,0.92]) # change locations of lower left and upper right of inset axes

ax.patch.set_alpha(0.5)

ax.plot(cum_sum_time_standardized_array, [row[1] for row in sorted_alaskan_peninsula_water_losses_and_depths], "g--", label = "Jakob")

ax.plot(cum_sum_array_uvolc, [row[1] for row in uvolc_losses_and_depths], label = "upper_volcs")
ax.plot(cum_sum_array_lvolc, [row[1] for row in lvolc_losses_and_depths], label = "lower_volcs")
ax.plot(cum_sum_array_dike, [row[1] for row in dike_losses_and_depths], "p--", label = "dikes")
ax.plot(cum_sum_array_gabbros, [row[1] for row in gabbros_losses_and_depths], label = "gabbros")
ax.plot(cum_sum_array_mantle, [row[1] for row in mantle_losses_and_depths], "r--", label = "mantle")

ax.yaxis.set_inverted(True)  # inverted axis with autoscaling
ax.legend()
ax.xaxis()
ax.yaxis()
# ax.set_xticks(np.linspace(0,10,8), "Tg/MYr/m")



ax.set_xticks([])
ax.set_yticks([])
ax.spines['bottom'].set_color('red')
ax.spines['top'].set_color('red')
ax.spines['right'].set_color('red')
ax.spines['left'].set_color('red')
ax.spines['bottom'].set_linewidth(1)
ax.spines['top'].set_linewidth(1)
ax.spines['right'].set_linewidth(1)
ax.spines['left'].set_linewidth(1)

fig.savefig(output_folder / "abers_alaskan_peninsula_comparison")

for splitting up contribtions to cumulative sum:

get 5 different cumulative sums, varying number of layers contributing to those unsorted cuulative sums
plot depth vs. cumulative sum curves for all 5



1. binning
2. generalzing
3. sediments
4. hirizontal vs. vertical depth logging
5. abstract